In [2]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc


# ============================================================
# CONFIG
# ============================================================

DATA_PATH = "cleaned.csv"
TIME_COL = "interval_start"
FUEL_COL = "fuel_type"
VALUE_COL = "generation_mw"

APP_TITLE = "Generation Dashboard"
SUBTITLE = "ERCOT-style layout for historical generation visuals"

FUEL_ORDER = [
    "BIOMASS", "COAL", "GAS", "GAS-CC", "HYDRO",
    "NUCLEAR", "OTHER", "SOLAR", "WIND", "WSL"
]

FUEL_COLORS = {
    "BIOMASS": "#1f77b4",
    "COAL": "#ff7f0e",
    "GAS": "#2ca02c",
    "GAS-CC": "#d62728",
    "HYDRO": "#9467bd",
    "NUCLEAR": "#8c564b",
    "OTHER": "#e377c2",
    "SOLAR": "#7f7f7f",
    "WIND": "#bcbd22",
    "WSL": "#17becf",
}

FUEL_LABELS = {
    "BIOMASS": "Biomass",
    "COAL": "Coal",
    "GAS": "Natural Gas",
    "GAS-CC": "Natural Gas Combined Cycle",
    "HYDRO": "Hydroelectric",
    "NUCLEAR": "Nuclear",
    "OTHER": "Other",
    "SOLAR": "Solar",
    "WIND": "Wind",
    "WSL": "WSL",
}


# ============================================================
# DATA LOADING / PREP
# ============================================================

def load_data(path: str | Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find data file: {path.resolve()}\n"
            "Update DATA_PATH to point to your cleaned.csv file."
        )

    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    if "Date" not in df.columns:
        raise ValueError("cleaned.csv must contain a 'Date' column.")

    # rename Date -> interval_start so the rest of the app stays consistent
    df = df.rename(columns={"Date": TIME_COL})

    # convert wide daily file to long format
    fuel_columns = [c for c in df.columns if c not in [TIME_COL, "total_all_fuels"]]

    if not fuel_columns:
        raise ValueError("No fuel columns found in cleaned.csv.")

    df = df.melt(
        id_vars=[TIME_COL],
        value_vars=fuel_columns,
        var_name=FUEL_COL,
        value_name=VALUE_COL
    )

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df[FUEL_COL] = df[FUEL_COL].astype(str).str.upper().str.strip()
    df[VALUE_COL] = pd.to_numeric(df[VALUE_COL], errors="coerce")

    df = df.dropna(subset=[TIME_COL, FUEL_COL, VALUE_COL]).copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)

    return df


def filter_complete_periods(df: pd.DataFrame) -> pd.DataFrame:
    """
    Daily-total version:
    - keep full days
    - keep months with at least 95% of days present
    - keep years with 12 qualifying months
    """
    out = df.copy()
    out["date"] = out[TIME_COL].dt.floor("D")

    day_counts = out.groupby("date")[FUEL_COL].nunique()
    expected_fuels = max(1, out[FUEL_COL].nunique())
    complete_days = day_counts[day_counts >= max(1, int(expected_fuels * 0.8))].index

    out = out[out["date"].isin(complete_days)].copy()

    complete_day_df = pd.DataFrame({"date": pd.Index(complete_days)})
    complete_day_df["month"] = complete_day_df["date"].dt.to_period("M")

    month_counts = complete_day_df.groupby("month").size()
    month_expected = pd.Series(
        {m: m.days_in_month for m in month_counts.index},
        dtype=float
    )
    complete_months = month_counts[month_counts >= month_expected * 0.95].index

    out["month"] = out[TIME_COL].dt.to_period("M")
    out = out[out["month"].isin(complete_months)].copy()

    complete_month_df = pd.DataFrame({"month": pd.Index(complete_months)})
    complete_month_df["year"] = complete_month_df["month"].astype(str).str[:4].astype(int)

    year_counts = complete_month_df.groupby("year").size()
    complete_years = year_counts[year_counts >= 12].index

    out["year"] = out[TIME_COL].dt.year
    out = out[out["year"].isin(complete_years)].copy()

    return out.reset_index(drop=True)


def compute_kpis_for_year(df_year: pd.DataFrame) -> dict:
    total_ts_year = (
        df_year.groupby(TIME_COL, as_index=False)[VALUE_COL]
        .sum()
        .rename(columns={VALUE_COL: "total_mw"})
    )

    daily_year = (
        df_year.assign(date=df_year[TIME_COL].dt.floor("D"))
        .groupby("date", as_index=False)[VALUE_COL]
        .sum()
        .rename(columns={VALUE_COL: "generation_mwh"})
    )

    total_by_fuel_year = (
        df_year.groupby(FUEL_COL, as_index=False)[VALUE_COL]
        .sum()
        .rename(columns={VALUE_COL: "generation_mwh"})
        .sort_values("generation_mwh", ascending=False)
    )

    peak_mw = total_ts_year["total_mw"].max() if not total_ts_year.empty else np.nan
    avg_mw = total_ts_year["total_mw"].mean() if not total_ts_year.empty else np.nan
    total_mwh = daily_year["generation_mwh"].sum() if not daily_year.empty else np.nan
    top_fuel = total_by_fuel_year.iloc[0][FUEL_COL] if not total_by_fuel_year.empty else None

    return {
        "peak_mw": peak_mw,
        "avg_mw": avg_mw,
        "total_mwh": total_mwh,
        "top_fuel": top_fuel,
    }


def prepare_aggregates(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    dfc = filter_complete_periods(df)

    total_ts = (
        dfc.groupby(TIME_COL, as_index=False)[VALUE_COL]
        .sum()
        .rename(columns={VALUE_COL: "total_mw"})
    )

    daily = (
        dfc.assign(date=dfc[TIME_COL].dt.floor("D"))
        .groupby("date", as_index=False)[VALUE_COL]
        .sum()
        .rename(columns={VALUE_COL: "generation_mwh"})
    )

    monthly_fuel = (
        dfc.assign(month=dfc[TIME_COL].dt.to_period("M").dt.to_timestamp())
        .groupby(["month", FUEL_COL], as_index=False)[VALUE_COL]
        .sum()
        .rename(columns={VALUE_COL: "generation_mwh"})
    )

    monthly_total = (
        monthly_fuel.groupby("month", as_index=False)["generation_mwh"]
        .sum()
    )

    yearly_fuel = (
        dfc.assign(year=dfc[TIME_COL].dt.year)
        .groupby(["year", FUEL_COL], as_index=False)[VALUE_COL]
        .sum()
        .rename(columns={VALUE_COL: "generation_mwh"})
    )

    yearly_total = yearly_fuel.groupby("year", as_index=False)["generation_mwh"].sum()
    yearly_share = yearly_fuel.merge(yearly_total, on="year", suffixes=("", "_total"))
    yearly_share["share_pct"] = (
        100 * yearly_share["generation_mwh"] / yearly_share["generation_mwh_total"]
    )

    # replacement for old hourly profile, since daily totals do not have hour-level detail
    avg_monthly_profile = (
        dfc.assign(month_num=dfc[TIME_COL].dt.month)
        .groupby(["month_num", FUEL_COL], as_index=False)[VALUE_COL]
        .mean()
        .rename(columns={VALUE_COL: "avg_mw"})
    )

    fuel_time = (
        dfc.groupby([TIME_COL, FUEL_COL], as_index=False)[VALUE_COL]
        .sum()
        .rename(columns={VALUE_COL: "mw"})
    )

    total_by_fuel = (
        dfc.groupby(FUEL_COL, as_index=False)[VALUE_COL]
        .sum()
        .rename(columns={VALUE_COL: "generation_mwh"})
        .sort_values("generation_mwh", ascending=False)
    )

    years = sorted(dfc["year"].dropna().unique().tolist())
    kpis_by_year = {}

    for year in years:
        df_year = dfc[dfc["year"] == year].copy()
        kpis_by_year[int(year)] = compute_kpis_for_year(df_year)

    latest_timestamp = total_ts[TIME_COL].max() if not total_ts.empty else None
    default_year = max(years) if years else None

    return {
        "clean": dfc,
        "total_ts": total_ts,
        "daily": daily,
        "monthly_fuel": monthly_fuel,
        "monthly_total": monthly_total,
        "yearly_fuel": yearly_fuel,
        "yearly_share": yearly_share,
        "avg_monthly_profile": avg_monthly_profile,
        "fuel_time": fuel_time,
        "total_by_fuel": total_by_fuel,
        "latest_timestamp": latest_timestamp,
        "years": years,
        "default_year": default_year,
        "kpis_by_year": kpis_by_year,
    }


# ============================================================
# FORMATTING HELPERS
# ============================================================

def fmt_number(x: float, suffix: str = "") -> str:
    if pd.isna(x):
        return "N/A"

    abs_x = abs(x)
    if abs_x >= 1_000_000_000:
        return f"{x / 1_000_000_000:.2f}B{suffix}"
    if abs_x >= 1_000_000:
        return f"{x / 1_000_000:.2f}M{suffix}"
    if abs_x >= 1_000:
        return f"{x / 1_000:.1f}K{suffix}"
    return f"{x:.0f}{suffix}"


def make_card(title: str, figure: go.Figure) -> dbc.Card:
    return dbc.Card(
        [
            dbc.CardHeader(
                html.Div(title, style={"fontWeight": "600", "fontSize": "16px"}),
                style={
                    "backgroundColor": "#0f172a",
                    "color": "white",
                    "borderBottom": "1px solid #1e293b",
                },
            ),
            dbc.CardBody(
                dcc.Graph(
                    figure=figure,
                    config={"displayModeBar": False},
                    style={"height": "100%"},
                ),
                style={"padding": "0.5rem"},
            ),
        ],
        style={
            "height": "100%",
            "borderRadius": "12px",
            "boxShadow": "0 2px 10px rgba(0,0,0,0.08)",
            "border": "1px solid #e5e7eb",
            "overflow": "hidden",
        },
    )


def kpi_card_component(title_id: str, value_id: str) -> dbc.Card:
    return dbc.Card(
        dbc.CardBody(
            [
                html.Div(id=title_id, style={"fontSize": "13px", "color": "#64748b"}),
                html.Div(id=value_id, style={"fontSize": "28px", "fontWeight": "700", "color": "#0f172a"}),
            ]
        ),
        style={
            "borderRadius": "12px",
            "boxShadow": "0 2px 10px rgba(0,0,0,0.08)",
            "border": "1px solid #e5e7eb",
        },
    )


def style_figure(fig: go.Figure, title: str, show_legend: bool = True) -> go.Figure:
    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            y=0.97,
            font=dict(size=16)
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        hovermode="x unified",
        margin=dict(l=40, r=20, t=110, b=40),
        font=dict(family="Arial, sans-serif", size=12, color="#0f172a"),
        showlegend=show_legend,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.03,
            xanchor="center",
            x=0.5,
            title=None,
            font=dict(size=10),
            bgcolor="rgba(255,255,255,0.85)"
        ),
    )

    fig.update_xaxes(showgrid=True, gridcolor="#e5e7eb", linecolor="#cbd5e1", ticks="outside")
    fig.update_yaxes(showgrid=True, gridcolor="#e5e7eb", linecolor="#cbd5e1", ticks="outside", zeroline=False)
    return fig


# ============================================================
# FIGURE BUILDERS
# ============================================================

def fig_total_generation(total_ts: pd.DataFrame) -> go.Figure:
    fig = px.line(
        total_ts,
        x=TIME_COL,
        y="total_mw",
        labels={TIME_COL: "Date", "total_mw": "Generation"}
    )
    fig.update_traces(line=dict(width=1.3, color="#2563eb"))
    return style_figure(fig, "Total Generation Over Time", show_legend=False)


def fig_daily_generation(daily: pd.DataFrame) -> go.Figure:
    fig = px.line(
        daily,
        x="date",
        y="generation_mwh",
        labels={"date": "Date", "generation_mwh": "Generation"}
    )
    fig.update_traces(line=dict(width=1.6, color="#0ea5e9"))
    return style_figure(fig, "Daily Total Generation", show_legend=False)


def fig_monthly_total(monthly_total: pd.DataFrame) -> go.Figure:
    fig = px.line(
        monthly_total,
        x="month",
        y="generation_mwh",
        markers=True,
        labels={"month": "Month", "generation_mwh": "Generation"}
    )
    fig.update_traces(line=dict(width=2, color="#1d4ed8"))
    return style_figure(fig, "Monthly Total Generation", show_legend=False)


def fig_monthly_fuel(monthly_fuel: pd.DataFrame) -> go.Figure:
    fig = px.bar(
        monthly_fuel,
        x="month",
        y="generation_mwh",
        color=FUEL_COL,
        category_orders={FUEL_COL: FUEL_ORDER},
        color_discrete_map=FUEL_COLORS,
        labels={"month": "Month", "generation_mwh": "Generation", FUEL_COL: "Fuel"},
    )
    fig.update_layout(barmode="stack")
    return style_figure(fig, "Monthly Generation by Fuel Type", show_legend=True)


def fig_yearly_share(yearly_share: pd.DataFrame) -> go.Figure:
    fig = px.bar(
        yearly_share,
        x="year",
        y="share_pct",
        color=FUEL_COL,
        category_orders={FUEL_COL: FUEL_ORDER},
        color_discrete_map=FUEL_COLORS,
        labels={"year": "Year", "share_pct": "Share (%)", FUEL_COL: "Fuel"},
    )
    fig.update_layout(barmode="stack")
    return style_figure(fig, "Fuel Mix Share by Year", show_legend=True)


def fig_avg_monthly_profile(avg_monthly_profile: pd.DataFrame) -> go.Figure:
    month_labels = {
        1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
        5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
        9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
    }

    plot_df = avg_monthly_profile.copy()
    plot_df["month_label"] = plot_df["month_num"].map(month_labels)

    fig = px.line(
        plot_df,
        x="month_label",
        y="avg_mw",
        color=FUEL_COL,
        category_orders={
            "month_label": list(month_labels.values()),
            FUEL_COL: FUEL_ORDER
        },
        color_discrete_map=FUEL_COLORS,
        labels={"month_label": "Month", "avg_mw": "Average Daily Generation", FUEL_COL: "Fuel"},
    )
    return style_figure(fig, "Average Monthly Generation Profile by Fuel Type", show_legend=True)


def fig_fuel_over_time(fuel_time: pd.DataFrame) -> go.Figure:
    fig = px.area(
        fuel_time,
        x=TIME_COL,
        y="mw",
        color=FUEL_COL,
        category_orders={FUEL_COL: FUEL_ORDER},
        color_discrete_map=FUEL_COLORS,
        labels={TIME_COL: "Date", "mw": "Generation", FUEL_COL: "Fuel"},
    )
    return style_figure(fig, "Generation by Fuel Type Over Time", show_legend=True)


def fig_total_by_fuel(total_by_fuel: pd.DataFrame) -> go.Figure:
    fig = px.bar(
        total_by_fuel,
        x=FUEL_COL,
        y="generation_mwh",
        color=FUEL_COL,
        category_orders={FUEL_COL: FUEL_ORDER},
        color_discrete_map=FUEL_COLORS,
        labels={FUEL_COL: "Fuel Type", "generation_mwh": "Generation"},
    )
    return style_figure(fig, "Total Generation by Fuel Type", show_legend=False)


# ============================================================
# APP
# ============================================================

df = load_data(DATA_PATH)
data = prepare_aggregates(df)

app = Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    title=APP_TITLE,
)

latest_text = (
    pd.to_datetime(data["latest_timestamp"]).strftime("%Y-%m-%d")
    if data["latest_timestamp"] is not None else "N/A"
)

app.layout = dbc.Container(
    fluid=True,
    style={"backgroundColor": "#f8fafc", "minHeight": "100vh", "padding": "0"},
    children=[
        html.Div(
            [
                html.Div(
                    [
                        html.H2(APP_TITLE, style={"margin": "0", "fontWeight": "700"}),
                        html.Div(SUBTITLE, style={"opacity": 0.9, "fontSize": "14px"}),
                    ]
                ),
                html.Div(
                    [
                        html.Div("Last Updated", style={"fontSize": "12px", "opacity": 0.85}),
                        html.Div(latest_text, style={"fontWeight": "600"}),
                    ],
                    style={"textAlign": "right"},
                ),
            ],
            style={
                "backgroundColor": "#0f172a",
                "color": "white",
                "padding": "18px 24px",
                "display": "flex",
                "justifyContent": "space-between",
                "alignItems": "center",
            },
        ),

        dbc.Container(
            fluid=True,
            style={"padding": "20px"},
            children=[
                dcc.Tabs(
                    id="year-tabs",
                    value=data["default_year"],
                    children=[
                        dcc.Tab(
                            label=str(year),
                            value=year,
                            style={
                                "padding": "10px 18px",
                                "fontWeight": "600",
                                "border": "1px solid #e5e7eb",
                                "backgroundColor": "#ffffff",
                            },
                            selected_style={
                                "padding": "10px 18px",
                                "fontWeight": "700",
                                "border": "1px solid #cbd5e1",
                                "backgroundColor": "#e2e8f0",
                                "color": "#0f172a",
                            },
                        )
                        for year in data["years"]
                    ],
                ),

                html.Div(style={"height": "16px"}),

                dbc.Row(
                    [
                        dbc.Col(kpi_card_component("peak-title", "peak-value"), md=3),
                        dbc.Col(kpi_card_component("avg-title", "avg-value"), md=3),
                        dbc.Col(kpi_card_component("energy-title", "energy-value"), md=3),
                        dbc.Col(kpi_card_component("fuel-title", "fuel-value"), md=3),
                    ],
                    className="g-3",
                ),

                html.Div(style={"height": "14px"}),

                dbc.Row(
                    [
                        dbc.Col(make_card("Total Generation", fig_total_generation(data["total_ts"])), md=6),
                        dbc.Col(make_card("Daily Total Generation", fig_daily_generation(data["daily"])), md=6),
                    ],
                    className="g-3",
                ),

                html.Div(style={"height": "14px"}),

                dbc.Row(
                    [
                        dbc.Col(make_card("Monthly Total Generation", fig_monthly_total(data["monthly_total"])), md=6),
                        dbc.Col(make_card("Total Generation by Fuel", fig_total_by_fuel(data["total_by_fuel"])), md=6),
                    ],
                    className="g-3",
                ),

                html.Div(style={"height": "14px"}),

                dbc.Row(
                    [
                        dbc.Col(make_card("Monthly Generation by Fuel", fig_monthly_fuel(data["monthly_fuel"])), md=6),
                        dbc.Col(make_card("Fuel Mix Share by Year", fig_yearly_share(data["yearly_share"])), md=6),
                    ],
                    className="g-3",
                ),

                html.Div(style={"height": "14px"}),

                dbc.Row(
                    [
                        dbc.Col(make_card("Average Monthly Profile", fig_avg_monthly_profile(data["avg_monthly_profile"])), md=6),
                        dbc.Col(make_card("Generation by Fuel Over Time", fig_fuel_over_time(data["fuel_time"])), md=6),
                    ],
                    className="g-3",
                ),
            ],
        ),
    ],
)


@app.callback(
    Output("peak-title", "children"),
    Output("peak-value", "children"),
    Output("avg-title", "children"),
    Output("avg-value", "children"),
    Output("energy-title", "children"),
    Output("energy-value", "children"),
    Output("fuel-title", "children"),
    Output("fuel-value", "children"),
    Input("year-tabs", "value"),
)
def update_kpis(selected_year):
    kpi = data["kpis_by_year"].get(int(selected_year), {})

    peak_mw = kpi.get("peak_mw", np.nan)
    avg_mw = kpi.get("avg_mw", np.nan)
    total_mwh = kpi.get("total_mwh", np.nan)
    top_fuel = kpi.get("top_fuel", None)

    return (
        f"Peak Generation ({selected_year})",
        fmt_number(peak_mw),
        f"Average Generation ({selected_year})",
        fmt_number(avg_mw),
        f"Total Energy ({selected_year})",
        fmt_number(total_mwh),
        f"Top Fuel by Energy ({selected_year})",
        FUEL_LABELS.get(str(top_fuel), str(top_fuel)) if top_fuel else "N/A",
    )


if __name__ == "__main__":
    app.run(debug=False, host="127.0.0.1", port=8054, use_reloader=False)